# 🧪 Advanced Feature Engineering Pipeline

Welcome to the **Feature Engineering Pipeline**. In our primary notebook, we completed the core data cleaning, handled anomalies, and established our base clean dataset.

The sole purpose of this notebook is to move beyond raw baseline metrics and extract deeper predictive signals (Alpha) from our historical data. Instead of forcing our downstream models to learn complex, non-linear market dynamics from scratch, we use this space to explicitly engineer advanced momentum, volatility, and trend indicators. This mathematical enrichment layer directly exposes the underlying structural mechanics of the Bitcoin market to our future models.

---

## 🎯 Primary Objectives

* **Information Extraction:** Transform simple base data points (Price and Volume) into mathematical representations of market velocity and trend strength.
* **Multi-Scale Momentum Tracking:** Implement short-term and long-term indicators (like EMAs and RSI) to capture shifting market regimes.
* **Volatility Quantization:** Isolate pricing spreads and rolling standard deviations to provide downstream algorithms with clean risk boundaries.
* **Data Integrity Checks:** Ensure all engineered indicators are correctly aligned chronologically and completely free of lookahead bias.

## 📥 Phase 2.1: Dataset Ingestion & Verification

In this step, we initialize our programming environment and load our cleaned baseline dataset to begin the feature engineering process.

---

### 💻 Code Actions (What the Script Does)

#### 1. Library Ingestion
* The code imports `pandas` as the core framework for manipulating data matrices and tables.
* It imports Google Colab's native `files` and `drive` modules to securely mount your external cloud storage directly into the active server runtime environment.

#### 2. File Path Mapping & Loading
* The script targets the explicit storage path where our immutable data artifact is stored: `.../Bitcoin_Project/Clean_Dataset.csv`.
* It uses `pd.read_csv()` to parse the file back into a working operational DataFrame (`df`).

#### 3. Structural Integrity Check (`df.head()`)
* The code calls `.head()` to output the first 5 rows of the dataframe to visually confirm that all columns, index headers, and numerical values are perfectly intact and formatted correctly before any new transformations are applied.

---

### 📋 Columns Retrieved in This Step

By running this initial code block, we confirm the following baseline columns are loaded and ready for feature engineering:

* `timestamp` (The index date)
* `close` (Daily closing price)
* `volume` (Baseline traded volume)
* `log_volume` (Log-transformed volume signal)
* `returns_scaled` (Robust-scaled return metrics)
* `buy_pressure` (Taker/Maker volume aggression ratio)
* `next_day_return` (Our continuous lookahead target variable)
* `pressure_bin` (Categorical buyer pressure classification)
* `day_sin` / `day_cos` (Cyclical day-of-the-week coordinates)
* `month_sin` / `month_cos` (Cyclical month-of-the-year coordinates)

In [ ]:
import pandas as pd
from google.colab import files
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
#import clean_Dataset
data_path = "/content/drive/MyDrive/Colab Notebooks/Bitcoin_Project/Clean_Dataset.csv"
df = pd.read_csv(data_path)
df.head()

,timestamp,close,volume,log_volume,returns_scaled,buy_pressure,next_day_return,pressure_bin,day_sin,day_cos,month_sin,month_cos
0,2018-01-02 00:00:00+00:00,14675.11,20078.092111,19.449289,3.315394,0.565110,0.557149,Bullish (>51%),0.781831,0.623490,0.0,1.0
1,2018-01-03 00:00:00+00:00,14919.51,15905.667639,19.279837,0.557149,0.565519,0.306989,Bullish (>51%),0.974928,-0.222521,0.0,1.0
2,2018-01-04 00:00:00+00:00,15059.54,21329.649574,19.561016,0.306989,0.594516,4.328231,Bullish (>51%),0.433884,-0.900969,0.0,1.0
3,2018-01-05 00:00:00+00:00,16960.39,23251.491125,19.727179,4.328231,0.574011,0.205961,Bullish (>51%),-0.433884,-0.900969,0.0,1.0
4,2018-01-06 00:00:00+00:00,17069.79,18571.457508,19.549554,0.205961,0.592693,-1.870547,Bullish (>51%),-0.974928,-0.222521,0.0,1.0


## 🛠️ Phase 2.2: Trend & Momentum Feature Engineering

In this step, we explicitly engineer trend-following and momentum metrics to capture the macro moving baselines of the Bitcoin market.

---

### 💻 Code Actions & New Engineered Columns

The execution of this script successfully injects **three new mathematical columns** directly derived from our baseline closing price:

1. **`ema_12` (Short-term Trend):** Tracks the exponential moving average price over a 12-day rolling span.
2. **`ema_26` (Long-term Trend):** Tracks the exponential moving average price over a 26-day rolling span.
3. **`ema_gap` (The Market Mood Indicator):** A custom ratio tracking the localized distance between the current asset price and its 12-day trend:
   $$\text{ema\_gap} = \frac{\text{close} - \text{ema\_12}}{\text{ema\_12}}$$

---

### 🧠 The Core Logic: Why We Added These Features

Based on our architectural design, we introduce these moving variations to provide our machine learning models with explicit context regarding market regimes:

* **Speed of Reaction:** Unlike a Simple Moving Average (SMA), an Exponential Moving Average (EMA) reacts much faster to sudden daily price changes because it assigns higher mathematical weight to the most recent data points.
* **Trend Phase Identification:** * **Uptrend:** If the closing price stays cleanly above the EMA lines, the market is broadly recognized as being in a bullish daily phase.
  * **Cross-over Signals:** When the short-term `ema_12` crosses above the long-term `ema_26`, it triggers a classic "Golden Cross" signature indicating strong momentum expansion.
* **Quantifying Overextensions via `ema_gap`:** By mapping the distance between the spot price and its underlying trend, the model can instantly recognize when the price is getting unsustainably "over-extended" relative to its historical mean.

---

### ✂️ Data Warm-up & Pipeline Cleaning

* **The Action:** Because exponential formulas require initialization histories to calibrate and become accurate, the first few days of calculations are structurally unstable.
* **The Solution:** The code programmatically slices away the first 26 uncalibrated rows using `df.iloc[26:]`, ensuring our down-stream algorithms train *exclusively* on stable, mature historical trend indicators.

In [ ]:
# 1. Create the 12-day and 26-day EMA (The Daily Trend Foundation)
df['ema_12'] = df['close'].ewm(span=12, adjust=False).mean()
df['ema_26'] = df['close'].ewm(span=26, adjust=False).mean()

# 2. Calculate the "EMA Gap" (The Market Mood Indicator)
# This shows how "over-extended" the price is compared to the 12-day average
df['ema_gap'] = (df['close'] - df['ema_12']) / df['ema_12']

# 3. EMA Warm-up & Cleaning
# Because the first few days of an EMA are less accurate, we drop the first 26 rows
# to ensure the model only trains on "mature" trend data.
df = df.iloc[26:].copy()

# Final check to see the new columns
print(df[['close', 'ema_12', 'ema_26', 'ema_gap']].head())

       close        ema_12        ema_26   ema_gap
26  11879.95  11716.034783  12544.286690  0.013991
27  11251.00  11644.490970  12448.487676 -0.033792
28  10237.51  11428.032359  12284.711552 -0.104176
29  10285.10  11252.196612  12136.592177 -0.085947
30   9224.52  10940.246364  11920.883127 -0.156827


## 🛠️ Phase 2.3: Relative Strength Index (RSI) Feature Engineering

In this step, we engineer a classic momentum oscillator to capture the speed and change of price movements in the Bitcoin market.

---

### 💻 Code Actions & New Engineered Column

The execution of this script creates a custom mathematical function to inject **one new momentum feature** directly into our dataset:

* **`rsi` (Relative Strength Index):** A normalized metric calculated over a rolling 14-day tracking period that captures the ratio of positive price changes (gains) to negative price changes (losses). It bounds the asset's internal velocity strictly between a scale of `0` and `100`.

---

### 🧠 The Core Logic: "Overbought" vs. "Oversold" Market States

Based on the core logic mapped above our function, we introduce this oscillator to give our machine learning models explicit boundaries for market extremes and neutral trends:

* **`RSI > 70` (Overbought Zone):** Indicates that the asset price has been rising very fast. Mathematically, the market might be reaching a state of structural **"exhaustion,"** signaling to the AI that a downward price correction or macro reversal has a higher probability of occurring.
* **`RSI < 30` (Oversold Zone):** Indicates that the asset price has been falling very fast. This suggests the market might be heavily **"undervalued,"** meaning a directional bounce upward is likely to follow.
* **`RSI ~ 50` (Neutral Zone):** Indicates that the market is coasting inside a stable trend devoid of extreme momentum, letting the model know there is no current structural overextension in either direction.

In [ ]:
def calculate_rsi(series,period=14):
  delta = series.diff()
  gain = (delta.where(delta > 0, 0)).ewm(alpha=1/period, adjust=False).mean()
  loss = (-delta.where(delta < 0, 0)).ewm(alpha=1/period, adjust=False).mean()
  rs = gain/loss
  return 100 - (100/(1+rs))

df['rsi'] = calculate_rsi(df['close'])
print(df[['close','rsi']].head(10))

       close        rsi
26  11879.95        NaN
27  11251.00   0.000000
28  10237.51   0.000000
29  10285.10   3.108432
30   9224.52   1.780290
31   8873.03   1.544726
32   9199.96  13.066791
33   8184.81   9.391532
34   6939.99   6.847972
35   7652.14  20.168710


## 🛠️ Phase 2.4: Volatility (Market Nervousness) Feature Engineering

In this step, we shift our focus from short-term market "noise" to institutional volatility by engineering a rolling metric that measures the health and stability of the underlying trend.

---

### 💻 Code Actions & New Engineered Column

The execution of this script calculates how much the asset's price "swings" away from its average, injecting **one new risk feature** into our dataset:

* **`volatility_7d`:** A rolling financial indicator that computes the 7-day (one full week) standard deviation of our `returns_scaled` feature vector.

---

### 🧠 The Core Logic: Why the Model Needs Volatility Context

Based on the strategic logic detailed above our code, we introduce this rolling window to prevent our machine learning models from getting "blind-sided" by sudden market regime changes:

* **High Volatility Signals:** Indicates that the price is making massive daily moves (such as during a vertical bull run or a "Black Swan" liquidation event). This provides the AI with immediate context to say: *"The price is rising, but volatility is exploding—this move might be unsustainable."*
* **Low Volatility Signals:** Indicates that the price is consolidating within a tight, quiet trading range. In historical asset cycles, this signature often precedes a massive directional "expansion" breakout move.

By providing this one-week lookback context, the algorithm can clearly distinguish between a steady, healthy trend and a chaotic, high-risk trading environment.

In [ ]:
# caculate 24 hours Rolling Volatility
df['volatility_7d'] = df['returns_scaled'].rolling(window=7).std()
print(f"length of dataset before dropping:{len(df)}")
print(f"Volatility calculated. New Column: 'volatility_7d'")
print(df[['returns_scaled', 'volatility_7d']].tail())

length of dataset before dropping:2969
Volatility calculated. New Column: 'volatility_7d'
      returns_scaled  volatility_7d
2990        0.494509       1.212074
2991       -0.048828       0.872672
2992        0.819116       0.913632
2993       -0.356995       0.696144
2994        0.163549       0.634613


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2969 entries, 26 to 2994
Data columns (total 17 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   timestamp        2969 non-null   object 
 1   close            2969 non-null   float64
 2   volume           2969 non-null   float64
 3   log_volume       2969 non-null   float64
 4   returns_scaled   2969 non-null   float64
 5   buy_pressure     2969 non-null   float64
 6   next_day_return  2969 non-null   float64
 7   pressure_bin     2969 non-null   object 
 8   day_sin          2969 non-null   float64
 9   day_cos          2969 non-null   float64
 10  month_sin        2969 non-null   float64
 11  month_cos        2969 non-null   float64
 12  ema_12           2969 non-null   float64
 13  ema_26           2969 non-null   float64
 14  ema_gap          2969 non-null   float64
 15  rsi              2968 non-null   float64
 16  volatility_7d    2963 non-null   float64
dtypes: float64(15

## 🛠️ Phase 2.5: Price Gap (Recent Momentum) & Final Feature Cleaning

In this final engineering block, we isolate daily momentum to capture recent price velocity, and perform a master trim to guarantee all indicator data fields are fully calculated and aligned.

---

### 💻 Code Actions & New Engineered Column

The execution of this script introduces a final short-term indicator and structures the final matrix shape:

* **`daily_momentum`:** A continuous feature generated by running `.pct_change()` on the closing price to track the single-day percentage velocity from yesterday to today.
* **`df_final` Creation:** Slices out uncalibrated startup rows and consolidates our engineering workbook into a definitive 18-column predictive matrix.

---

### 🧠 The Core Logic: The Psychology of Momentum

Based on the psychological framework detailed above our script, daily momentum serves as a vital trend-setting guide for machine learning models:

* **The FOMO Factor:** Because Bitcoin is a highly emotional asset, a strong positive daily velocity often creates a structural "Fear of Missing Out" (FOMO) effect. This psychological momentum carries over into the next trading session, driving further near-term buying action.
* **The Stop-Loss Cascade:** Conversely, a sharp daily drop often triggers automated stop-loss orders and panic selling, rapidly driving the asset downward.
* **Feature vs. Target Distinction:** It is crucial to distinguish past momentum from future return vectors:
  * **`daily_momentum` (Input Feature):** Measures what *already* happened from yesterday to today.
  * **`next_day_return` (Target Label):** Tracks what *will* happen from today to tomorrow.
  The AI leverages today's historical velocity to predict tomorrow's target direction.

---

### ✂️ Master Data Trimming: Why We Drop the First 40 Rows

To clear out incomplete data across all our indicators simultaneously, the code executes a strict structural slice: `df.iloc[40:].copy()`. This specific threshold is required to account for the unique initialization lookback needs of each engineered feature:

* **`daily_momentum`** requires **1 day** of historical context.
* **`volatility_7d`** requires **7 days** of rolling context.
* **`rsi`** requires **14 days** of rolling context.
* **`EMA_26`** requires a minimum of **26 days** of price data to stabilize its exponential weights.

By dropping the first 40 rows, we clear away all unstable `NaN` artifacts and initial boundary noise across every indicator at once. This guarantees that our final dataset shape (**2,929 rows by 18 columns**) consists entirely of synchronized, mature, and mathematically sound market data.

In [ ]:
#caculate price gap
df['daily_momentum']=df['close'].pct_change()

# We drop the first 40 rows to clean up ALL features at once:
# EMA (Needs 26), RSI (Needs 14), Volatility (Needs 7), Momentum (Needs 1)
df_final = df.iloc[40:].copy()

# display final dataset
print("--- Final Cleaned Dataset (Last 5 Days) ---")
display_cols = ['close', 'daily_momentum', 'rsi', 'ema_gap', 'volatility_7d', 'next_day_return']
print(df_final[display_cols].tail())

print(f"\nFinal dataset shape: {df_final.shape}")

--- Final Cleaned Dataset (Last 5 Days) ---
         close  daily_momentum        rsi   ema_gap  volatility_7d  \
2990  69960.64        0.014834  50.902098  0.020684       1.212074   
2991  69894.00       -0.000953  50.737782  0.016628       0.872672   
2992  71590.01        0.024265  54.742011  0.034723       0.913632   
2993  70880.82       -0.009906  52.809011  0.020630       0.696144   
2994  71250.68        0.005218  53.726712  0.021875       0.634613   

      next_day_return  
2990        -0.048828  
2991         0.819116  
2992        -0.356995  
2993         0.163549  
2994         0.708207  

Final dataset shape: (2929, 18)


## 🛠️ Phase 2.6: Target Engineering — The "Future Shift" & Leakage Prevention

In this step, we finalize our core training metrics by constructing our predictive target variable and handling structural data boundaries to prevent artificial machine learning results.

---

### 💻 Code Actions & Final Matrix Definition

The execution of this script explicitly configures the predictive alignment between our inputs and outputs:

* **`df_final['target']`:** Created by shifting the `returns_scaled` metric upward by exactly one row using `.shift(-1)`.
* **Boundary Cleanup:** Because shifting the final row upward leaves no tomorrow data available for the last entry, it creates a trailing `NaN`. The script drops this incomplete entry via `.dropna(subset=['target'])` to ensure clean training execution.
* **Final Length Check:** Confirms the finalized, fully synchronized training matrix contains **2,928 valid operational rows**.

---

### 🧠 The Core Logic: Preventing Data Leakage in Daily Forecasting

Based on the trading rules mapped out above our function, this chronological adjustment resolves a fatal machine learning pitfall:

* **The Problem of Data Leakage:** In a live daily forecasting application, we use the complete historical data available *when today's candle closes* ($T$) to predict the unknown price movement of *tomorrow* ($T+1$). If we accidentally try to predict the `returns_scaled` variable on its own current row, the model will essentially cheat. It will recognize that today's close price was higher than yesterday's and simply guess a positive return, creating an illusion of **99% testing accuracy** that completely fails and loses money in live markets because it is "predicting the past".
* **The Solution:** We explicitly force our features ($X$) and our target ($y$) to sit on separate, sequential time horizons:
  * **The Features ($X$):** Highly descriptive indicators (such as `buy_pressure`, `log_volume`, cyclical temporal encodings, `ema_gap`, `rsi`, `volatility_7d`, and `daily_momentum`) remain completely fixed at Today ($T$).
  * **The Target ($y$):** Represents the robustly scaled return coordinate of Tomorrow ($T+1$).

By implementing this shift, we force our models to look for real lead indicators—identifying complex predictive patterns in today's volatility and momentum metrics that possess authentic mathematical power to determine tomorrow's market direction.

In [ ]:
#define target
df_final['target'] = df_final['returns_scaled'].shift(-1)
#drop last row
df_final = df_final.dropna(subset=['target'])
print(f"Final length of dataset for training: {len(df_final)}")
df_final.head()
#we will use thi code later
#features = [buy_pressure', 'log_volume', 'day_sin', 'day_cos', 'month_sin', 'month_cos', 'ema_gap', 'rsi', 'volatility_7d','daily_momentum']
# x = df[features]
#y = df['target']

Final length of dataset for training: 2928


,timestamp,close,volume,log_volume,returns_scaled,buy_pressure,next_day_return,pressure_bin,day_sin,day_cos,month_sin,month_cos,ema_12,ema_26,ema_gap,rsi,volatility_7d,daily_momentum,target
66,2018-03-09 00:00:00+00:00,9227.00,64112.291407,20.156016,-0.181754,0.486122,-1.719880,Bearish (<49%),-0.433884,-0.900969,0.866025,0.5,10256.918248,10284.560978,-0.100412,40.478996,1.495415,-0.004815,-1.719880
67,2018-03-10 00:00:00+00:00,8770.22,37180.012857,19.648009,-1.719880,0.474444,2.979629,Bearish (<49%),-0.974928,-0.222521,0.866025,0.5,10028.195440,10172.387572,-0.125444,37.319008,1.165226,-0.049505,2.979629
68,2018-03-11 00:00:00+00:00,9533.57,44325.973386,19.815665,2.979629,0.477152,-1.468156,Bearish (<49%),-0.781831,0.623490,0.866025,0.5,9952.099219,10125.067752,-0.042054,45.040479,1.966092,0.087039,-1.468156
69,2018-03-12 00:00:00+00:00,9131.34,42230.777930,19.795820,-1.468156,0.458663,0.054289,Bearish (<49%),0.000000,1.000000,0.866025,0.5,9825.828570,10051.458289,-0.070680,42.097698,1.950749,-0.042191,0.054289
70,2018-03-13 00:00:00+00:00,9150.00,40191.409358,19.721836,0.054289,0.494047,-3.702306,Neutral,0.781831,0.623490,0.866025,0.5,9721.854944,9984.683601,-0.058822,42.286088,1.914004,0.002044,-3.702306


## 🛠️ Phase 2.7: Data Persistence — Saving the "Gold" Dataset

In this concluding step of our pipeline, we save our completed feature matrix to ensure a stable and consistent boundary for all upcoming model training scripts.

---

### 💻 Code Actions (What the Script Does)

The execution of this short code block exports and downloads our finalized dataframe:

1. **`df_final.to_csv()`:** This method serializes the internal Python DataFrame object into a flat, raw text standard CSV file named `"Processed_Dataset.csv"`.
2. **`index = False`:** By explicitly setting the index parameter to False, we instruct pandas *not* to create an unlabelled, redundant integer sequence column in the file, keeping the data structured and clean.
3. **`files.download()`:** This utilizes Google Colab's native file module to trigger an automated localized browser download stream, saving the processed CSV file directly to your hardware storage.

---

### 🧠 The Core Logic: Why Data Persistence Matters

Based on the architectural setup of our project, saving this definitive dataset artifact accomplishes two major production goals:

* **Eliminating Redundant Calculations:** Feature engineering operations (such as calculating rolling indicators, log scaling, and temporal encodings) consume processing time and compute power. Saving this finalized snapshot means future model development notebooks can instantly load this file without needing to repeat any mathematical transformations.
* **Ensuring Deterministic Training:** It guarantees that our separate neural network models (like LSTMs) and tree-based ensembles (like XGBoost) are training on the exact same input features ($X$) and target labels ($y$) across different experimental sessions.

In [ ]:
processed_dataset = df_final.to_csv("Processed_Dataset.csv",index = False )
files.download("Processed_Dataset.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>